# Metacatalog NED-LVS cross-match

Cross-match the OVRO-LWA metacatalog against the latest [NED Local Volume Sample](https://ned.ipac.caltech.edu/NED::LVS/) (Cook et al. 2023).

**Primary metric:** fraction of Blue-associated metacatalog rows with a NED-LVS galaxy within the combined beam / galaxy angular size.

**Complementary metrics:** NED-LVS recovery in the metacatalog Dec footprint, over-splitting (one galaxy matched by multiple meta rows), and multiplicity (multiple NED-LVS galaxies per meta row).

Matching uses metacatalog primary `RA`/`DEC` with `BMAJ_match` via `resolve_bmaj`. NED-LVS rows use the fiducial `Diam` major-axis (arcsec) when available, otherwise a 20″ default radius.

Unlike VLSSR QA, not every radio detection is expected to have a NED-LVS host — treat unmatched meta rows as candidates for hostless or non-galaxy emission.

**Run cells in order.**

In [ ]:
from pathlib import Path

import pandas as pd

from lwa_catalog import CatalogLayout, read_metacatalog
from lwa_catalog.analyze import (
    NedlvsMatchConfig,
    load_nedlvs_catalog,
    match_catalog_to_nedlvs,
    select_blue_associated_rows,
    summarize_nedlvs_match,
)
from lwa_catalog.constants import NEDLVS_DEFAULT_PATH

# --- operator config ---
CATALOG_DIR = Path("/fast/claw/metacatalog_coaddR-0.75_subband")
NEDLVS_PATH = NEDLVS_DEFAULT_PATH  # latest NED-LVS FITS on lwacalim09
RUN_LST_MERGED_PASS = True

layout = CatalogLayout(CATALOG_DIR)
config = NedlvsMatchConfig(catalog_path=NEDLVS_PATH)

print("CATALOG_DIR =", layout.root.resolve())
print("NEDLVS_PATH =", Path(NEDLVS_PATH).resolve())
print("RUN_LST_MERGED_PASS =", RUN_LST_MERGED_PASS)

## Load metacatalog

In [ ]:
metacatalog = read_metacatalog(layout)
blue_meta = select_blue_associated_rows(metacatalog)

print(f"metacatalog rows: {len(metacatalog)}")
print(f"Blue-associated rows: {len(blue_meta)}")

## Load NED-LVS

Loads ~2.1M rows from the FITS table (takes ~15–30 s on lwacalim09).

In [ ]:
nedlvs = load_nedlvs_catalog(NEDLVS_PATH)
print(f"NED-LVS rows: {len(nedlvs):,}")
print(f"rows with Diam: {(nedlvs['Diam_arcsec'] > 0).sum():,}")
nedlvs.head()

## Cross-match against NED-LVS

In [ ]:
result = match_catalog_to_nedlvs(metacatalog, nedlvs=nedlvs, config=config)
print(summarize_nedlvs_match(result))

## Diagnostics

In [ ]:
meta_flags = result.meta_flags
nedlvs_flags = result.nedlvs_flags

print("NED-LVS hits per meta (value_counts):")
display(meta_flags["n_nedlvs"].value_counts().sort_index())

print("\nMeta rows with no NED-LVS host (first 20):")
display(meta_flags.loc[~meta_flags["matched"]].head(20))

oversplit_cols = [
    c
    for c in [
        "nedlvs_pos",
        "RA",
        "DEC",
        "objname",
        "DistMpc",
        "n_meta",
        "meta_ids",
        "oversplit",
    ]
    if c in nedlvs_flags.columns
]
print("\nOver-split NED-LVS rows (multiple meta matches, first 20):")
display(nedlvs_flags.loc[nedlvs_flags["oversplit"], oversplit_cols].head(20))

## NED-LVS recovery vs distance

For each `DistMpc` threshold, recovery is the fraction of footprint NED-LVS galaxies at or closer than that distance with ≥1 metacatalog match.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np

dist = pd.to_numeric(nedlvs_flags["DistMpc"], errors="coerce")
nedlvs_matched = nedlvs_flags["n_meta"].gt(0)
ok = np.isfinite(dist.to_numpy(dtype=float)) & (dist > 0)

dist_vals = dist.loc[ok].to_numpy()
matched = nedlvs_matched.loc[ok].astype(int).to_numpy()
order = np.argsort(dist_vals)
dist_vals = dist_vals[order]
matched = matched[order]

suffix_matched = np.cumsum(matched)
suffix_total = np.arange(1, len(dist_vals) + 1, dtype=float)
recovery = suffix_matched / suffix_total
overall_recovery = float(result.summary["nedlvs_recovery"])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(dist_vals, recovery, drawstyle="steps-post")
ax.axhline(
    overall_recovery,
    color="C1",
    ls="--",
    lw=1,
    label=f"All footprint ({overall_recovery:.3f})",
)
ax.set_xscale("log")
ax.set_xlabel("NED-LVS distance threshold (Mpc)")
ax.set_ylabel("NED-LVS recovery")
ax.set_ylim(0, 1.01)
ax.set_title(f"NED-LVS recovery vs distance ({len(dist_vals):,} footprint galaxies)")
ax.legend(loc="lower right")
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

## Optional: LST-merged Blue pass

Ad-hoc pre-fusion check. Set `RUN_LST_MERGED_PASS = True` in the config cell.

In [ ]:
if RUN_LST_MERGED_PASS:
    from lwa_catalog.io import read_lst_merged

    lst_blue = read_lst_merged(layout, "Blue")
    lst_config = NedlvsMatchConfig(catalog_path=NEDLVS_PATH, target="lst_merged_blue")
    lst_result = match_catalog_to_nedlvs(lst_blue, nedlvs=nedlvs, config=lst_config)
    print("LST-merged Blue pass:")
    print(summarize_nedlvs_match(lst_result))
else:
    print("Skipping LST-merged Blue pass (RUN_LST_MERGED_PASS=False)")